# Дерево решений

Практические ячейки к слайдам презентации. Заголовки секций совпадают с заголовками слайдов.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)



## Бинарная классификация деревом `(viz)`


Двумерная задача: дерево строит оси-параллельные границы. Обучим дерево на двух признаках и нарисуем разбиение пространства.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

iris = load_iris()
X = iris.data[:, [0, 2]]  # sepal length, petal length
y = (iris.target != 0).astype(int)  # класс 0 vs остальные

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
print(f"Train accuracy: {clf.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {clf.score(X_test, y_test):.3f}")

# сетка для фона
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200),
)
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 5))
ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdYlBu")
ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", marker="o", label="класс 0", edgecolors="k", s=40)
ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:orange", marker="^", label="класс 1", edgecolors="k", s=40)
ax.set_xlabel("sepal length")
ax.set_ylabel("petal length")
ax.set_title("Бинарная классификация: оси-параллельные области")
ax.legend()
plt.tight_layout()
plt.show()


## Энтропия и индекс Джини `(viz)`


Три функции неоднородности для бинарного случая (p — доля класса 1).


In [ ]:
def misclassification(p):
    return 1 - np.maximum(p, 1 - p)

def gini(p):
    return 2 * p * (1 - p)

def entropy(p):
    eps = 1e-12
    q = 1 - p
    return -(p * np.log2(p + eps) + q * np.log2(q + eps))

p = np.linspace(0.001, 0.999, 300)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p, misclassification(p), label="Misclassification", lw=2)
ax.plot(p, entropy(p), label="Энтропия", lw=2)
ax.plot(p, gini(p), label="Джини", lw=2)
ax.set_xlabel("p — доля класса 1")
ax.set_ylabel("Impurity")
ax.set_title("Функции неоднородности (бинарный случай)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Энтропия и индекс Джини `(interactive)`


Подвиньте ползунок p — посмотрите, как меняются три критерия в одном узле.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

@widgets.interact(p=widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01, description="p"))
def show_impurities(p=0.5):
  values = {
      "Misclassification": misclassification(p),
      "Энтропия": entropy(p),
      "Джини": gini(p),
  }
  fig, ax = plt.subplots(figsize=(6, 3))
  ax.bar(values.keys(), values.values(), color=["#4c72b0", "#55a868", "#c44e52"])
  ax.set_ylim(0, 1.05)
  ax.set_ylabel("Impurity")
  ax.set_title(f"p = {p:.2f}")
  plt.tight_layout()
  plt.show()


## Оценка качества дерева `(example)`


На несбалансированных данных accuracy обманчива. Считаем precision, recall, F1 по классам.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X, y = make_classification(
    n_samples=1000, n_features=8, n_informative=5,
    weights=[0.9, 0.1], random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
print("Accuracy:", round(clf.score(X_test, y_test), 3))
print(classification_report(y_test, clf.predict(X_test), digits=3))


## Оценка качества дерева `(experiment)`


`class_weight='balanced'` меняет веса классов **при построении** дерева (impurity), а не при predict.


In [ ]:
for cw in [None, "balanced"]:
    m = DecisionTreeClassifier(class_weight=cw, random_state=42)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(f"\nclass_weight={cw!r}")
    print(classification_report(y_test, pred, digits=3))


## Переобучение и его решение `(experiment)`


Зависимость accuracy от глубины: train растёт, test часто образует «колокол».


In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=300, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

best_depth = depths[int(np.argmax(test_scores))]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(depths, train_scores, "o-", label="train")
ax.plot(depths, test_scores, "s-", label="test")
ax.axvline(best_depth, color="gray", ls="--", alpha=0.7)
ax.annotate(f"best depth={best_depth}", xy=(best_depth, max(test_scores)), xytext=(best_depth + 1, max(test_scores) - 0.05))
ax.fill_between(depths, 0, 1, where=[d > best_depth for d in depths], alpha=0.08, color="red", label="overfitting region")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.set_title("Переобучение дерева")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Переобучение и его решение `(interactive)`


Интерактивно: меняйте `max_depth` и смотрите границу решения на train/test.


In [ ]:
@widgets.interact(max_depth=widgets.IntSlider(value=3, min=1, max=12, description="depth"))
def plot_boundary(max_depth=3):
    m = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    m.fit(X_train, y_train)
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 150),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 150),
    )
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (X_, y_, title) in zip(axes, [(X_train, y_train, "train"), (X_test, y_test, "test")]):
        ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlBu")
        ax.scatter(X_[:, 0], X_[:, 1], c=y_, cmap="RdYlBu", edgecolors="k", s=25)
        ax.set_title(f"{title}, acc={m.score(X_, y_):.2f}")
    plt.suptitle(f"max_depth={max_depth}")
    plt.tight_layout()
    plt.show()


## Дерево решений в sklearn `(example)`


Базовый пайплайн: обучение, predict, основные гиперпараметры.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = DecisionTreeClassifier(max_depth=3, ccp_alpha=0.01, random_state=42)
clf.fit(X_train, y_train)

print("Параметры:", clf.get_params(deep=False))
print("Test accuracy:", round(clf.score(X_test, y_test), 3))

sample = X_test[:3]
print("Predict:", clf.predict(sample))
print("Proba:\n", np.round(clf.predict_proba(sample), 2))


## Интерпретируемость и важность признаков `(example)`


`feature_importances_` — суммарный вклад признака в уменьшение impurity (взвешенный по числу объектов в узлах).


In [ ]:
import pandas as pd

imp = pd.Series(clf.feature_importances_, index=wine.feature_names).sort_values(ascending=False)
print(imp.head())


## Интерпретируемость и важность признаков `(viz)`


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("importance")
ax.set_title("Важность признаков (wine)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## Дерево решений для регрессии `(example)`


В регрессии в листьях — среднее y. Критерий по умолчанию — MSE; для устойчивости к выбросам — `absolute_error`.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=200, n_features=1, noise=8, random_state=42)
# добавим выброс
y[5] += 80
y[10] -= 60

X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(X, y, s=25, alpha=0.7, label="данные")

for criterion, color in [("squared_error", "tab:blue"), ("absolute_error", "tab:green")]:
    reg = DecisionTreeRegressor(max_depth=3, criterion=criterion, random_state=42)
    reg.fit(X, y)
    ax.plot(X_plot, reg.predict(X_plot), lw=2, color=color, label=criterion)

ax.set_title("Регрессионное дерево: MSE vs MAE")
ax.legend()
plt.tight_layout()
plt.show()


## Визуализация деревьев в sklearn `(example)`


Графическое дерево и текстовые правила.


In [ ]:
from sklearn.tree import plot_tree, export_text

fig, ax = plt.subplots(figsize=(14, 8))
plot_tree(clf, feature_names=wine.feature_names, class_names=wine.target_names.astype(str), filled=True, ax=ax)
plt.tight_layout()
plt.show()

print(export_text(clf, feature_names=wine.feature_names))


## Границы решения: оси-параллельные прямоугольники `(viz)`


Три класса, глубина 3 — видны прямоугольные области и «лестница» границы.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data[:, [0, 1]]
y = iris.target

clf3 = DecisionTreeClassifier(max_depth=3, random_state=42)
clf3.fit(X, y)

x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
Z = clf3.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.contourf(xx, yy, Z, alpha=0.35, cmap="viridis")
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", edgecolors="k", s=35)
ax.set_xlabel(iris.feature_names[0])
ax.set_ylabel(iris.feature_names[1])
ax.set_title("Границы решения: 3 класса, depth=3")
plt.colorbar(scatter, ax=ax, ticks=[0, 1, 2])
plt.tight_layout()
plt.show()


## Какая предобработка нужна деревьям? `(example)`


Номинативный признак нельзя кодировать как 0,1,2 — дерево будет сравнивать «цвет < 1.5». Нужен One-Hot.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

df = pd.DataFrame({
    "size": ["S", "M", "L", "M", "S"],
    "color": ["red", "blue", "green", "red", "blue"],
    "price": [10, 20, 30, 22, 12],
})
y = np.array([0, 1, 1, 1, 0])

# плохо: Label Encoding для color
bad_X = df.copy()
bad_X["color"] = bad_X["color"].map({"red": 0, "blue": 1, "green": 2})
bad = DecisionTreeClassifier(max_depth=2, random_state=42)
bad.fit(bad_X, y)
print("Плохое кодирование (Ordinal для color):")
print(export_text(bad, feature_names=list(bad_X.columns)))

# хорошо: One-Hot для color, порядковый size
pre = ColumnTransformer([
    ("size_ord", OrdinalEncoder(categories=[["S", "M", "L"]]), ["size"]),
    ("color_oh", OneHotEncoder(sparse_output=False), ["color"]),
], remainder="passthrough")

pipe = Pipeline([
    ("prep", pre),
    ("tree", DecisionTreeClassifier(max_depth=2, random_state=42)),
])
pipe.fit(df, y)
print("\nС One-Hot для color — правила осмысленны.")
print("Колонки после prep:", pipe.named_steps["prep"].get_feature_names_out())
